# Checkpoint 2: Pneumonia Detection from Chest X-Rays

Binary image classification — **NORMAL** vs **PNEUMONIA** chest X-rays.

**Dataset:** [Kaggle – Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) by Paul Mooney

**Sections:**
1. Dataset download & setup
2. Data exploration and visualisation
3. Preprocessing and augmentation
4. Baseline CNN model training
5. Evaluation and metrics

> **Tip:** Go to **Runtime → Change runtime type → T4 GPU** for faster training.

In [ ]:
!pip install -q tensorflow numpy pandas matplotlib scikit-learn seaborn pillow kaggle


## 0 · Dataset Setup

### Option A – Kaggle API *(recommended)*
1. Log into Kaggle → **Account** → **Settings** → **API** → **Create New Token** (downloads `kaggle.json`)
2. Run the next cell — it will prompt you to upload `kaggle.json`

### Option B – Manual upload
1. Download the ZIP from Kaggle manually and upload it to `/content/`
2. Skip the Kaggle cell and run: `!unzip -q /content/archive.zip -d /content/`

In [ ]:
import os
from google.colab import files

print('Upload your kaggle.json API token:')
uploaded = files.upload()

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/ --unzip
print('Dataset ready.')


## Configuration

All paths and hyperparameters are defined here. Edit `EPOCHS` or `BATCH_SIZE` if needed.

In [ ]:
from pathlib import Path

# Paths
BASE_DIR    = Path('/content')
DATASET_DIR = BASE_DIR / 'chest_xray'
OUTPUT_DIR  = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_DIR = OUTPUT_DIR / 'results'
MODELS_DIR  = OUTPUT_DIR / 'models'

# Hyperparameters
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 10
RANDOM_SEED = 42

for d in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  Dataset : {DATASET_DIR}')
print(f'  Outputs : {OUTPUT_DIR}')
print(f'  Image size={IMAGE_SIZE}  Batch size={BATCH_SIZE}  Epochs={EPOCHS}')


## 1 · Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

EXPECTED_SPLITS  = ('train', 'val', 'test')
EXPECTED_CLASSES = ('NORMAL', 'PNEUMONIA')
VALID_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

def _list_images(directory):
    return [f for f in directory.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS]

# Verify dataset structure
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR}. Run the download cell first.')

missing = [DATASET_DIR / s / c
           for s in EXPECTED_SPLITS for c in EXPECTED_CLASSES
           if not (DATASET_DIR / s / c).exists()]
if missing:
    print('WARNING - missing folders:')
    for p in missing:
        print(f'  {p}')
else:
    print('Dataset structure OK')

# Count images per split/class
rows = [{'split': s, 'class': c, 'count': len(_list_images(DATASET_DIR / s / c))}
        for s in EXPECTED_SPLITS for c in EXPECTED_CLASSES]
summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULTS_DIR / 'dataset_summary.csv', index=False)
print(summary_df.pivot(index='split', columns='class', values='count').fillna(0).astype(int))

# Class distribution bar plot
plt.figure(figsize=(8, 5))
sns.barplot(data=summary_df, x='split', y='count', hue='class')
plt.title('Class Distribution by Dataset Split')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

# Sample image grid
MAX_PER_CLASS = 4
fig, axes = plt.subplots(2, MAX_PER_CLASS, figsize=(3 * MAX_PER_CLASS, 6), squeeze=False)
for row_idx, cls in enumerate(EXPECTED_CLASSES):
    samples = _list_images(DATASET_DIR / 'train' / cls)[:MAX_PER_CLASS]
    for col_idx in range(MAX_PER_CLASS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            ax.imshow(Image.open(samples[col_idx]).convert('L'), cmap='gray')
            ax.set_title(cls, fontsize=9)
        ax.axis('off')
plt.suptitle('Sample Training Images', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'example_images_grid.png', dpi=150)
plt.show()


## 2 · Preprocessing & Data Augmentation

Each image is resized to **224x224**, normalised to **[0, 1]**, and converted from grayscale to 3 channels.  
Training images are augmented with random rotation (5%), zoom (10%), horizontal flip, and contrast shift (10%).

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def load_images_from_directory(directory, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',
        color_mode='grayscale',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=RANDOM_SEED,
    )

def resize_and_normalize(images):
    return tf.cast(images, tf.float32) / 255.0

def ensure_three_channel(images):
    if images.shape[-1] == 1:
        return tf.image.grayscale_to_rgb(images)
    if images.shape[-1] > 3:
        return images[..., :3]
    return images

def get_augmentation_layer():
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomContrast(0.1),
    ], name='augmentation')

def preprocess_dataset(dataset, augment=False, augmentation_layer=None):
    def _preprocess(images, labels):
        images = resize_and_normalize(images)
        images = ensure_three_channel(images)
        if augment and augmentation_layer is not None:
            images = augmentation_layer(images, training=True)
        return images, labels
    return dataset.map(_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def create_datasets():
    train_raw = load_images_from_directory(DATASET_DIR / 'train', shuffle=True)
    val_raw   = load_images_from_directory(DATASET_DIR / 'val',   shuffle=False)
    test_raw  = load_images_from_directory(DATASET_DIR / 'test',  shuffle=False)
    class_names = train_raw.class_names
    aug_layer   = get_augmentation_layer()
    train_ds = preprocess_dataset(train_raw, augment=True, augmentation_layer=aug_layer)
    val_ds   = preprocess_dataset(val_raw)
    test_ds  = preprocess_dataset(test_raw)
    return train_ds, val_ds, test_ds, class_names

print('Preprocessing functions ready.')


## 3 · Baseline CNN Model

3 x (Conv2D + MaxPooling) blocks with 32/64/128 filters, followed by Flatten, Dropout(0.3), Dense(128), Dropout(0.2), and a sigmoid output.  
Compiled with **Adam** optimizer, **binary crossentropy** loss, and metrics: accuracy, precision, recall.

In [ ]:
def build_baseline_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

model = build_baseline_model()
model.summary()


## 4 · Training

Early stopping monitors `val_loss` with patience=3 and restores the best weights automatically.

In [ ]:
import numpy as np

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

train_ds, val_ds, test_ds, class_names = create_datasets()
print(f'Class index mapping: {class_names}')

model = build_baseline_model()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)


In [ ]:
# Save training history CSV
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / 'training_history.csv', index=False)

# Plot accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150)
plt.show()

# Save model
model.save(MODELS_DIR / 'baseline_cnn.keras')
print(f'Model saved to {MODELS_DIR}/baseline_cnn.keras')


## 5 · Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix

# Evaluate on test set
test_results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'accuracy', 'precision', 'recall']
metrics      = dict(zip(metric_names, test_results))

print('Test Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

with open(RESULTS_DIR / 'baseline_metrics.txt', 'w') as f:
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}', file=f)

# Collect predictions for confusion matrix
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)
    y_true.extend(labels.numpy().flatten().astype(int))
    y_pred.extend(preds)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Baseline CNN')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

print(f'All outputs saved to {OUTPUT_DIR}')
